# Determine overlapping regions

In [10]:
# Load packages
suppressPackageStartupMessages({
    library(GenomicRanges)
    library(data.table)
    library(dplyr)
})

In [6]:
# I/O
main = '/rds/project/rds-SDzz0CATGms/users/bt392/07_scRNA_CutTag/01_bulk'

In [14]:
bed = fread(file.path(main, 'output.bed'))[,c(1:4)] %>%
    setnames(paste0('V', 1:4), c('chr', 'start', 'end', 'sample')) %>%
    .[,sample:=lapply(strsplit(sample, '_'), `[`, 2)]

In [47]:
# Split bed in list per target
split_bed = lapply(unique(bed$sample), function(x){
    tmp = bed[sample==x]
})
names(split_bed) = unique(bed$sample)

# Convert to Granges objects
for(i in names(split_bed)){
    tmp = makeGRangesFromDataFrame(split_bed[[i]]) # Creates GRanges object from DF/DT
    assign(i,get("tmp")) # Saves GRanges object to name of TF
}

# Also make list w all GRanges
Granges = lapply(names(split_bed), function(x){
    makeGRangesFromDataFrame(split_bed[[x]])
    })

In [75]:
# find regions that are bound by all factors
for(i in 1:4){#length(unique(bed$sample))){
    if(i==1){
        overlap = Granges[[i]]
    }else{
        overlap = overlap[queryHits(findOverlaps(overlap, Granges[[i]]))]
    }
}
overlap

GRanges object with 1245 ranges and 0 metadata columns:
         seqnames              ranges strand
            <Rle>           <IRanges>  <Rle>
     [1]     chr1     4623261-4623661      *
     [2]     chr1   10325492-10325892      *
     [3]     chr1   10325492-10325892      *
     [4]     chr1   13154433-13154833      *
     [5]     chr1   13574073-13574473      *
     ...      ...                 ...    ...
  [1241]     chrX 145362535-145362935      *
  [1242]     chrX 162284230-162284630      *
  [1243]     chrX 164899820-164900220      *
  [1244]     chrX 167205786-167206186      *
  [1245]     chrX 169106339-169106739      *
  -------
  seqinfo: 22 sequences from an unspecified genome; no seqlengths

GRanges object with 4845 ranges and 0 metadata columns:
         seqnames              ranges strand
            <Rle>           <IRanges>  <Rle>
     [1]     chr1     3451625-3452025      *
     [2]     chr1     3915199-3915599      *
     [3]     chr1     4496277-4496677      *
     [4]     chr1     4623261-4623661      *
     [5]     chr1   10325492-10325892      *
     ...      ...                 ...    ...
  [4841]     chrX 166316792-166317192      *
  [4842]     chrX 167201329-167201729      *
  [4843]     chrX 167205786-167206186      *
  [4844]     chrX 169060789-169061189      *
  [4845]     chrX 169106339-169106739      *
  -------
  seqinfo: 22 sequences from an unspecified genome; no seqlengths

In [56]:
        overlap = findOverlaps(Granges[[1]], Granges[[2]])


In [58]:
Granges[[1]][queryHits(overlap)]

GRanges object with 4845 ranges and 0 metadata columns:
         seqnames              ranges strand
            <Rle>           <IRanges>  <Rle>
     [1]     chr1     3451625-3452025      *
     [2]     chr1     3915199-3915599      *
     [3]     chr1     4496277-4496677      *
     [4]     chr1     4623261-4623661      *
     [5]     chr1   10325492-10325892      *
     ...      ...                 ...    ...
  [4841]     chrX 166316792-166317192      *
  [4842]     chrX 167201329-167201729      *
  [4843]     chrX 167205786-167206186      *
  [4844]     chrX 169060789-169061189      *
  [4845]     chrX 169106339-169106739      *
  -------
  seqinfo: 22 sequences from an unspecified genome; no seqlengths